> **Save your own copy first.** Unless this notebook is already in your own Google Drive, Colab does not keep your changes. Go to **File → Save a copy in Drive** and work in the copy that opens in a new tab. You find it later in Google Drive, in *Colab Notebooks*.

> **Important:** Please run the cell below to set up the environment.

In [ ]:
%%capture
#Keep cell

import os

# palettable is needed by CDAIML.py (colours of the neural-network drawing)
try:
  import palettable
except:
  !pip install palettable
  import palettable


if not(os.path.exists('snsdata.csv')):
  print('Downloading snsdata.csv\n')
  !wget https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/snsdata.csv

if not(os.path.exists('spotify-dataset.csv')):
  print('Downloading spotify-dataset.csv\n')
  !wget https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/spotify-dataset.csv

if not(os.path.exists('CDAIML.py')):
  print('Downloading CDAIML.py\n')
  !wget https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/CDAIML.py


from CDAIML import DrawNN
from CDAIML import display_youtube

![Cultural Data Analysis · Week 3: Structured Data and Machine Learning](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week3-banner.png)

# Machine Learning

*Lecture notebook* · Code-along: [goto4711.github.io/cda-teaching-tools/code-along/week3](https://goto4711.github.io/cda-teaching-tools/code-along/week3/) · Tools: [how a computer sees an image](https://goto4711.github.io/cda-teaching-tools/tools/images/), [neural network training](https://goto4711.github.io/cda-teaching-tools/tools/neural-net/)

---

> **Goals** · This notebook is about one question: **how does a machine learn from data?** We follow the same steps twice:
> - **Supervised learning:** a neural network learns from labelled examples to predict whether a song is danceable.
> - **Unsupervised learning:** k-means finds groups of teenagers with similar interests, without any labels.
>
> At every step, we also ask: **who decided?** Machines learn from data, but people choose the data, the labels, the features and the settings. Those choices shape what the machine learns.

Machine learning is much less scary than science fiction wants us to believe. Machines do not learn because they want to understand the meaning of life, but to complete a particular task, and they still learn mostly from human examples. *Supervised* learning uses data that people have labelled; *unsupervised* learning finds patterns in unlabelled data, which people then have to interpret. Both work on *structured data*: tables like a spreadsheet, with one row per observation and one column per *feature* (https://www.ibm.com/think/topics/structured-vs-unstructured-data).

Run the cell below to load today's libraries.

In [ ]:
#Keep cell
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

> **Background: machine learning in our lives** (optional reading)
>
> Machines still learn largely from our examples, and they need large amounts of data to learn. This also makes machine learning an ethically difficult endeavour. It demands ever more data, which requires vast amounts of energy. Most aspects of our lives are recorded in gigantic data stores that machines can easily access. Governments, businesses and individuals record and report all manner of information, from the monumental to the mundane. As long as an activity can be transformed into a digital format, you can be certain that somebody will record it.
>
> Machine-learning algorithms are all around you. They have tried to predict the outcomes of elections and referenda, they identify spam messages, predict crime and natural disasters, target donors and voters, and have learned to drive cars, sometimes. They also frequently get it wrong, as many news articles will tell you.
>
> The ethics of machine learning develops more slowly than the technology, and laws and norms struggle to stay up to date. There is, for instance, an ongoing debate about how biased machine-learning algorithms are with regard to race and gender. Machine learning has also made it possible to identify people based on the region they live in, the products they buy, etc. As a machine-learning practitioner, you are often required to exclude data that is ethically problematic. This is not easy, as the connections are sometimes not obvious and might only be revealed after you have trained the machine.
>
> The methods we learn today have in common that they predict new observations from old observations: they are empirical and predictive, using models. Many steps of machine learning repeat themselves, and they are made easier by Python libraries such as scikit-learn (https://scikit-learn.org/), which does most of the hard work, and by ChatGPT and Gemini.

### The data science process

Machine learning is a process of repeatable steps. Schutt and O'Neil (2013) give an excellent overview:

![alt text](https://github.com/goto4711/cdai/blob/main/process.png?raw=true "The Art of Data Science ")

We collect data, explore and clean it (often 80% of the work), decide on a question, train a model on part of the data, and evaluate it on the rest. Then we improve and repeat.

> **Background: models, training and kinds of data** (optional reading)
>
> Just like humans, machines use data to generalise. In the words of machine learning, they form a *model*, which summarises the data and makes patterns in it explicit. Models can be (statistical) equations, figures like graphs or trees, rules or clusters. Machines don't choose the type of model: we choose it for them, when we analyse the task at hand and the available data.
>
> The computer learns to fit the model to the data in a process called *training*. Then we *test* the model in a separate process. The model does not include anything but what can be found in the data already. It can nevertheless be interesting, as it might surface connections we did not recognise before. Newton discovered gravity this way, if the myth is to be believed, by fitting equations (a model) to observations of falling apples. Gravity was always there, but it was observed for the first time in a model. Modelling is also never perfect. It generally involves some kind of bias or systematic error: Newton's laws are not as universal as he thought. Errors are not necessarily bad, as they can lead to a better model.
>
> In social and cultural data analysis, models often fall short of the performance we would like. Humans are difficult for computers, and their data is very noisy. People are inconsistent and report data wrongly, and some things are hard or impossible to measure. How do you quantify love? Online dating agencies still make a business out of predicting it. In history, we often lack data for whole periods, because records have been lost, and the data we have is full of missing values or follows diverse and contradictory standards. Finally, much data is so valuable that it is kept behind the walls of company servers or government agencies.
>
> So machine learning is not artificial intelligence yet, but a laborious collaboration between humans and machines that involves trying models and fighting with (bad) data.
>
> **Kinds of data.** Data describes a series of observations, the rows of a dataframe. Each observation is defined by its *features*, the columns. A feature measured in numbers is *numeric*, such as the ages of a group of students. A feature that takes one of a set of categories is *categorical* or *nominal*, such as the colour codes red, green and blue. *Ordinal* features are categories in an order: film ratings from 1 to 5 are ordinal, because it is not clear what lies between 2 and 3. DataCamp has a nice overview (https://www.datacamp.com/tutorial/categorical-data):
>
> ![alt text](https://github.com/goto4711/cdai/blob/main/D08t9.jpg?raw=true "Qualitative and Quantitative Data")
>
> Can you see how problematic certain characterisations can be?
>
> In *supervised* learning, an algorithm is given training data and learns a combination of features that predicts something, such as whether an earthquake will take place soon or whether a song is danceable. What we are trying to predict is called the *target variable* or *label*.

### How a machine sees similarity: feature spaces

Machines learn similarities as distances. Lantz (2013, *Machine Learning with R*) tastes 15 foods and records how crunchy and how sweet each is, on a scale from 1 to 10. In a *feature space* with one axis per feature, similar foods end up close together:

![alt text](https://github.com/goto4711/cdai/blob/main/lantz-2.png?raw=true "")

Now we taste a tomato for the first time. Is it a fruit or a vegetable? The machine decides by distance: which group is the tomato closest to?

![alt text](https://github.com/goto4711/cdai/blob/main/lantz-3.png?raw=true "")

> **Background: the feature space** (optional reading)
>
> Lantz's book (Lantz, B. (2013). *Machine Learning with R*. Packt Publishing) is old but still excellent, and it gives us the teenagers' dataset later on. Here is his feature space with a few example foods:
>
> ![alt text](https://github.com/goto4711/cdai/blob/main/lantz-1.png?raw=true "")
>
> Lantz observes that similar types of food end up close together: vegetables tend to be crunchy but not sweet, fruits tend to be sweet and either crunchy or not, and proteins tend to be neither (p. 68). Similarity is thus the distance between items in the feature space.
>
> For the tomato, the picture is not conclusive: we cannot see by eye which group it is closer to. We need an algorithm that measures the distances. You have just learned how a machine would think about the tomato, and which decisions it has to make: machines learn similarities in feature spaces using distances.

> **Who decided?** Someone chose to describe food by crunchiness and sweetness, on a scale from 1 to 10. With other features, such as colour or price, the tomato would land somewhere else.

---

# Supervised Machine Learning: Machine Dancing

Can a machine learn taste? For the philosopher Kant, a judgement of taste is subjective and yet demands that everyone agree with it (https://oll.libertyfund.org/titles/bernard-the-critique-of-judgement, §7). We try it with music: the machine should learn to predict whether a song is danceable. In the language of machine learning, this is a *classification* task with two classes.

## Step 1: Collect the data

Our data is a Kaggle copy of a dataset collected through Spotify's Web API: 114,000 tracks from 114 genres (https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset). Besides artist, title and genre, each track has *audio features* that Spotify's own algorithms computed from the music: danceability, energy, valence (how positive it sounds), acousticness, tempo and so on, mostly on a scale from 0 to 1. Spotify has since closed this part of its API, so nobody can collect these features today. The full list of columns: https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset.

Run the cell below to load the data.

In [ ]:
#Keep cell
music_df = pd.read_csv('spotify-dataset.csv')

Look at the data with `music_df.info()` and `music_df.head()`.

> **Background: the Spotify data** (optional reading)
>
> Kaggle (https://www.kaggle.com/datasets) is a famous repository of machine-learning datasets, among other things. Our dataset was collected with Spotify's Web API, across 114 genres (https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset). The creator did not document the technical details, but it follows the standard patterns of collecting Spotify data. The audio features were pre-computed by Spotify with a combination of signal processing and algorithms (https://developer.spotify.com/documentation/web-api/reference/get-audio-features). The most important columns:
> - *popularity*: 0 to 100, calculated by an algorithm, mostly from the total number of plays and how recent they are. Songs played a lot now score higher than songs played a lot in the past.
> - *duration_ms*: the length of the track in milliseconds.
> - *danceability*: how suitable a track is for dancing, based on tempo, rhythm stability, beat strength and overall regularity; 0.0 is least and 1.0 most danceable.
> - *energy*: 0.0 to 1.0, a perceptual measure of intensity and activity. Death metal has high energy, a Bach prelude low.
> - *loudness*: the overall loudness in decibels (dB).
> - *speechiness*: the presence of spoken words. Above 0.66, a track is probably entirely spoken (talk show, audio book, poetry); between 0.33 and 0.66, it may contain music and speech, as in rap.
> - *acousticness*: 0.0 to 1.0, confidence that the track is acoustic.
> - *instrumentalness*: the likelihood that a track contains no vocals ('ooh' and 'aah' count as instrumental).
> - *liveness*: the presence of an audience; above 0.8, the track is very likely live.
> - *valence*: 0.0 to 1.0, the musical positiveness. High valence sounds happy, cheerful or euphoric; low valence sad, depressed or angry.
> - *tempo*: the estimated tempo in beats per minute (BPM).
> - *key*, *mode* (major is 1, minor 0) and *time_signature* describe the musical structure; *track_genre* is the genre.
>
> As you can see, even music can be transformed into measurable features. You don't need to remember the details. This example is based on one of the previous years' data projects at the end of the semester, so feel inspired.
>
> The data comes as a CSV file (comma-separated values), perhaps the most common format of freely available structured data. Pandas' `pd.read_csv()` loads it. The file is in the same folder as the notebook, so we do not need a path; to learn more about paths, watch https://www.youtube.com/watch?v=0yIFEJ879lw.

> **Who decided?** Spotify's algorithms computed every audio feature, and we cannot see how. The dataset's creator chose 1,000 tracks per genre, and the genres themselves are Spotify's.

## Step 2: The label

`danceability` runs from 0.0 (least danceable) to 1.0 (most danceable). Plot its distribution with `sns.histplot(music_df['danceability'], bins=20)`.

For a classification, we need two classes: good or bad danceability. 0.6 splits the data roughly in half. `np.where` creates the new column `dance_label`: 1 for good, 0 for bad danceability. Run the cell below.

In [ ]:
#Keep cell
DANCEABILITY_THRESHOLD = 0.6 # Let's say songs with danceability >= 0.6 are "High Danceability"

music_df['dance_label'] = np.where(music_df['danceability'] >= DANCEABILITY_THRESHOLD, 1, 0)

Check the two classes with `music_df['dance_label'].value_counts()`.

What do the extremes sound like? The two cells below link to YouTube (click on the image: YouTube does not allow playing them inside the notebook).

In [ ]:
#Keep cell

least_danceable_example = "RWjxrp4C3xk"
display_youtube(least_danceable_example, video_title="Example of least danceable music")

In [ ]:
#Keep cell

most_danceable_example = "zVcMi4YCLWQ"
display_youtube(most_danceable_example, video_title="Example of most danceable music")

Which genres are the most danceable? Run the cell below.

In [ ]:
#Keep cell

plt.figure(figsize=(10, 6))
genre_counts = music_df['track_genre'].value_counts().nlargest(10)

avg_danceability_per_genre = music_df.groupby('track_genre')['danceability'].mean().sort_values(ascending=False).nlargest(10)
sns.barplot(x=avg_danceability_per_genre.values, y=avg_danceability_per_genre.index, hue = avg_danceability_per_genre.values, palette='magma', orient='h')
plt.title(f'Average Danceability for Top Genres')
plt.xlabel('Average Danceability Score')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

So, the genre 'kids', children's music, is the most danceable? In the workshop, we look into such questions.

> **Who decided?** The label is Spotify's danceability score, an algorithm's output, cut at 0.6. We chose the 0.6. The machine will learn to predict Spotify's score, not whether people dance.

## Step 3: Choose and prepare the features

The *features* are the columns the machine learns from. Run the cell below to choose them and copy them into `music_features_df`. `danceability` itself is left out: it is what we predict.

In [ ]:
#Keep cell

feature_cols = [
    'duration_ms', 'popularity', 'energy', 'key', # 'danceability' removed
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'tempo', 'time_signature', 'valence'
]

music_features_df = music_df[feature_cols].copy()

Remember the feature space: the machine compares songs by distance. If one feature is measured in milliseconds (`duration_ms`, up to millions) and another between 0 and 1, the large one dominates every distance. So we *normalise* all features to the same range from 0 to 1 with *min-max* normalisation: (value - minimum) / (maximum - minimum). Run the cell below to define it.

In [ ]:
#Keep cell

def normalize(x):
    return ((x - x.min()) / (x.max() - x.min()))

> **Background: normalisation** (optional reading)
>
> Normalisation is very important for all algorithms based on feature spaces, as the distances depend on how the features are measured. If some features have much larger values than others, the larger ones dominate the distances. This was no problem for the food data, as sweetness and crunchiness were both measured from 1 to 10. But suppose we added a measure from 1 to 1,000,000: it would dwarf the others.
>
> An example of min-max normalisation: a track has a `popularity` of 50, the maximum popularity is 100 and the minimum 20. The new value is (50-20)/(100-20) = 0.375. Can you see why the result is always between 0 and 1? We only normalise numeric features.
>
> `apply` is a powerful Pandas function that applies a function to every column of a dataframe.

Apply `normalize` to every column and add the label back. Type in:
```
music_features_normalized_df = music_features_df.apply(normalize)
music_features_normalized_df['dance_label'] = music_df['dance_label']
music_features_normalized_df.head()
```

> **Who decided?** We chose 13 features, including `popularity`, which measures how much a song is played on Spotify, not anything about the music. We also chose to give every feature the same weight.

## Step 4: Training and test data

We want to know how well the model predicts songs it has *not* seen. So we keep 20% of the songs aside as *test data* and train only on the other 80%. `train_test_split` shuffles the songs and splits them; `random_state=4711` makes the shuffle repeatable, so your results match this notebook. Run the cell below.

In [ ]:
#Keep cell
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(music_features_normalized_df, test_size=0.20, random_state=4711)
print(len(train_set), 'songs for training,', len(test_set), 'for testing')

> **A shortcut:** We normalised all songs before splitting them, so in principle the test songs helped decide how the training data was scaled. This is called *data leakage* (https://en.wikipedia.org/wiki/Leakage_(machine_learning)). The correct order is: split first, take the minimum and maximum from the training data only, and use them to scale both sets. We took the shortcut to keep the demo simple. Here it makes no difference, because the training songs already contain every extreme value, but in your own projects, split first.

> **Background: why training and test data?** (optional reading)
>
> We want to predict new things, so we need to know how the model behaves on data it does not know. If we had more music, we could collect it, apply our model and compare. But we cannot know about data we do not have. So we simulate it: we divide our data into a training dataset to build the model and a test dataset to find out how well the model predicts unknown data. This is the standard way of evaluating machine learning.
>
> Before splitting, the data is shuffled, so that all kinds of songs are evenly distributed in both datasets. You could also do this by hand with Pandas:
> ```
> train_set = music_features_normalized_df.sample(frac=0.80, random_state=4711)
> test_set = music_features_normalized_df.drop(train_set.index)
> ```
> `sample` draws a random 80% (`frac=0.80`), and `drop` removes those rows from the full data, which leaves the other 20%. The two methods select different songs, as each is random, but the sizes are the same. There are many ways to make training and test data: https://stackoverflow.com/questions/24147278/how-do-i-create-test-and-train-samples-from-one-dataframe-with-pandas.
>
> `.values` turns a Pandas column or dataframe into an array, which is what scikit-learn expects.

scikit-learn wants the features (`X`) and the label (`y`) separately, as arrays (`.values`). Type in:
```
X_train = train_set.loc[:, train_set.columns != 'dance_label'].values
y_train = train_set['dance_label'].values

X_test = test_set.loc[:, test_set.columns != 'dance_label'].values
y_test = test_set['dance_label'].values
```

## Step 5: The model, a neural network

A neural network is loosely inspired by the brain: layers of units connected by *weights*. The input units take the features of a song, the output unit gives the prediction, and the *hidden* layers in between do the work.

![alt text](https://github.com/goto4711/cdai/blob/main/neural-networks-2.png?raw=true "")

The network learns by feedback, called *backpropagation*: it predicts the training songs, compares its predictions with the labels, and adjusts its weights to make fewer mistakes, again and again. Try it with the neural network tool: https://goto4711.github.io/cda-teaching-tools/tools/neural-net/.

scikit-learn's `MLPClassifier` (a multi-layer perceptron) is such a network. As in the code-along, there are three steps: make a model, fit it, predict. Here we make the model and fit it; predicting follows in Step 6. Type in:
```
from sklearn.neural_network import MLPClassifier
model = MLPClassifier(hidden_layer_sizes=(20, 10), max_iter=300, random_state=4711)
model.fit(X_train, y_train)
```
`hidden_layer_sizes=(20, 10)` means two hidden layers with 20 and 10 units; `max_iter=300` stops after at most 300 rounds of feedback. Training takes a minute or two.

> **Background: neural networks** (optional reading)
>
> Neural networks are machines assembled in ways loosely similar to the brain's cells. Each neuron has a cell body with many dendrites (carrying information toward the cell body) and a single axon (carrying it away):
>
> ![alt text](https://github.com/goto4711/cdai/blob/main/neural-networks-1.png?raw=true "")
>
> But computers are not alive. They are not made of densely interconnected brain cells, each connected to perhaps 10,000 others. They store lots of data, rearrange it and need instructions for that. To this day, we do not fully understand how brains learn: they spontaneously put information together in astounding new ways. No computer currently comes close. A neural network simulates densely interconnected cells inside a computer, but it remains a mathematical equation: highly interconnected numbers in boxes that constantly change, which mean nothing to the computer itself.
>
> A typical network has anything from a few dozen to millions of artificial neurons, called *units*, arranged in *layers*. The *input units* take the data: one unit for each feature. The *output layer* gives the answer, in our case good or bad danceability. In between are one or more layers of *hidden units*, which form the majority of this artificial brain. Each connection between two units has a number, a *weight*, which can be positive (one unit excites another) or negative (it inhibits another). The higher the weight, the more influence one unit has on the other. Inputs come in from the left, activate the hidden units and feed out outputs on the right.
>
> Information also flows backwards. For a network to learn, there has to be feedback, just like when we humans learn: we compare what we tried to achieve with what we achieved and adjust. Networks do this with *backpropagation*. From the training data, we know the output we should get. We compare it with the network's output and change the weights, from the output units through the hidden units to the inputs, to get closer. Over time, the network learns until it reaches a stable state.
>
> `MLPClassifier` is a *multi-layer perceptron*: perceptrons are an older name for the units of a network (https://en.wikipedia.org/wiki/Perceptron), and with two hidden layers our network is multi-layered. The settings of the network, such as its layers and their sizes, are *hyperparameters*. Unlike the weights (the parameters), which are learned from data, hyperparameters are set by people before training, and poor choices cause many different problems (https://www.geeksforgeeks.org/machine-learning/hyperparameter-tuning/).

What has the network learned? Only weights: numbers on the connections between the units. Run the cell below to draw them (orange connections are positive, blue ones negative).

In [ ]:
#Keep cell

network_structure = [13, 20, 10, 1]
weights = model.coefs_

vis = DrawNN(
    network_structure,
    weights_list=weights,
    title=f"Large Network {network_structure}",
    max_linewidth=2.5,
    linewidth_base=0.1,
    linewidth_scale_tier1=3.0,
    linewidth_scale_tier2=6.0,
    weight_text_show_threshold=0.7 # Show fewer weight texts for clarity
)
vis.draw()

> **Who decided?** The size of the network (two layers, 20 and 10 units) and the number of rounds are *hyperparameters*: settings chosen by people before training, often by trying (https://en.wikipedia.org/wiki/Hyperparameter_(machine_learning)). Other choices give other networks.

## Step 6: Evaluate

Now predict the test songs, which the model has never seen, with `y_pred = model.predict(X_test)`.

A *confusion matrix* compares the predictions with the true labels. Run the cell below.

In [ ]:
#Keep cell
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_).plot()
plt.show()

> **Background: reading a confusion matrix** (optional reading)
>
> A confusion matrix compares the predictions with the true labels. Its four cells are the *true negatives* (bad songs predicted bad), *false positives* (bad songs predicted good), *false negatives* (good songs predicted bad) and *true positives* (good songs predicted good): https://en.wikipedia.org/wiki/False_positives_and_false_negatives and https://en.wikipedia.org/wiki/Confusion_matrix.

The diagonal holds the correct predictions: bad songs predicted bad (top left) and good songs predicted good (bottom right). *Accuracy* is their share of all test songs. Type in `(cm[0, 0] + cm[1, 1]) / cm.sum()`.

> **Background: accuracy** (optional reading)
>
> *Accuracy* is the number of correct predictions divided by the number of all predictions: the two cells on the diagonal, divided by the number of songs in the test data. Your number can be a little higher or lower than mine, depending on the random composition of the training and test data.
>
> About 79% is not bad, especially considering that music experts would probably not agree with each other to such a degree. In the workshop, we try to improve on it.
>
> That was it for supervised learning. It might seem complicated at first, but the same steps repeat again and again: get the data, clean it, create training and test data, run a model and evaluate it.

About 79% of the predictions are correct. Is that good? Compare it with a machine that has learned nothing and always says 'not danceable'. Type in `(y_test == 0).mean()`.

Guessing 'not danceable' every time is right for about 55% of the songs, so the network has learned something. But remember the label: it has learned to predict Spotify's score.

> **Who decided?** What counts as good enough? 79% may be impressive for a recommendation and unacceptable for a decision about people. And accuracy alone can hide which mistakes the model makes, as you will see with hate speech in Week 4.

---

# Unsupervised Machine Learning: Teenagers' Interests

A popular use of *clustering* is finding communities in social networks. Unlike supervised learning, there is no label: the machine groups the data by similarity, and people interpret the groups.

## Step 1: Collect the data

The data comes from Brett Lantz's book *Machine Learning with R* (2013, p. 279). It was compiled for sociological research on teenage identities: a random sample of 30,000 US high-school students with profiles on a social networking service in 2006, evenly spread over four graduation years (2006 to 2009). For each teenager, it records gender, age and number of friends. From the 500 most frequent words on the profiles, the researchers chose 36 words for five kinds of interest: extracurricular activities, fashion, religion, romance and antisocial behaviour. The data counts how often each word appears on each profile.

Run the cell below to load the data, and look at the first rows with `teens.head()`.

In [ ]:
#Keep cell
teens = pd.read_csv("snsdata.csv")

> **Background: the teenagers dataset** (optional reading)
>
> The dataset comes from the book by Lantz (2013), who discusses it on p. 279. It covers the relationships in a social networking service (SNS). It is a very early SNS dataset, but it is freely available, and it offers the same kinds of insights as recent data. All datasets of the book are collected at https://github.com/stedy/Machine-Learning-with-R-datasets/tree/master.
>
> The data was compiled for sociological research on teenage identities at the University of Notre Dame. It is a random sample of 30,000 US high-school students who had profiles on a well-known SNS in 2006, when it was a popular destination for US teenagers. So the profiles are likely a fairly wide cross-section of American teenagers in 2006. The sample covers four graduation years evenly (2006 to 2009): the senior, junior, second-year and first-year classes at the time.
>
> The full texts of the profiles were downloaded, and each teenager's gender, age and number of friends were recorded. A text-mining tool divided the rest of the page into words; you will learn how this works in Week 4. From the top 500 words, 36 were chosen to represent five categories of interest. For each person, the data counts how often each word appeared on the profile.

> **Who decided?** The researchers chose the 36 words and the five kinds of interest. Whatever teenagers wrote about that is not on the list, the machine will never see.

## Step 2: Clean the data

Real-life data is messy: values are missing or wrong. Count the genders, including the missing ones, with `teens['gender'].value_counts(dropna=False)`.

2,724 teenagers have no gender recorded, and the data knows only 'F' and 'M'. For the analysis later, we need a column that is 1 for female and 0 for male, and stays empty where gender is missing. `map()` translates each value with a dictionary. Type in `teens['female'] = teens['gender'].map({'F': 1, 'M': 0})`.

Now the ages. Always look at the smallest and largest values: type in `teens['age'].describe()`.

Teenagers aged 3 or 106? As a rule of thumb, we say that high-school students are between 13 and 19, and we mark all other ages as missing. Run the cell below (as in the code-along).

In [ ]:
#Keep cell
too_young = teens['age'] < 13
too_old = teens['age'] >= 20
teens.loc[too_young | too_old, 'age'] = np.nan
teens['age'].isna().sum()

That makes 5,523 missing ages. Deleting all these teenagers would lose too much data, so we fill in an estimate. The average of everyone would do, but we can do better: teenagers who graduate earlier are older. Check the average age per graduation year with `teens.groupby('gradyear')['age'].mean()`.

Now fill each missing age with the average of its graduation year. `transform('mean')` gives every row the average of its own group, so the averages line up with the ages. Type in:
```
year_mean = teens.groupby('gradyear')['age'].transform('mean')
teens['age'] = teens['age'].fillna(year_mean)
teens['age'].describe()
```

> **Background: missing values, dummy coding and imputation** (optional reading)
>
> Missing observations are a common problem in real-life data about society. In Python, they appear as `NaN`. `teens.info()` shows them too: columns with missing values have a lower non-null count.
>
> There are two basic strategies. The brute-force one removes all rows with missing values (`dropna()`). Here, we would lose too many teenagers. The other one fills them with estimates.
>
> **Gender.** We turned the gender column into numbers. This is called *dummy coding* and is typical of machine learning: a numerical variable represents subgroups (https://en.wikipedia.org/wiki/Dummy_variable_(statistics)). Our `female` column is 1 for 'F' and 0 for 'M'. We could add another column, `no_gender`, that is 1 where gender is missing, and use it to check whether certain communities tend not to record their gender. Pandas can create such columns in one go: `pd.get_dummies(teens, columns=['gender'])`. Please note that many datasets assume a binary division of gender, which is wrong.
>
> **Age.** Filling missing values with the average is called *mean imputation* and is used frequently: `teens['age'].fillna(teens['age'].mean())`. Pandas has many tools for missing data: https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html. We did better with some good old-fashioned human intelligence: we know the graduation year, and those who graduate earlier should be older. So we used the average age per graduation year.
>
> **groupby.** `groupby()` is essential knowledge in Pandas (https://pandas.pydata.org/docs/user_guide/groupby.html). It is at the heart of the split-apply-combine paradigm: it splits the data into groups, applies a function to each group, such as `mean()`, and combines the results. It requires lots of practice. With `transform`, every row gets the result of its own group, so it lines up with the original data. Another way is to `merge` the averages per year into the data (https://pandas.pydata.org/pandas-docs/stable/user_guide/merging.html), and then replace the missing ages.
>
> As long as you remember the kind of steps we have taken, you can fill one column's missing values by using another column as a reference.

> **Who decided?** We decided that teenagers are between 13 and 19, and that a missing age is the average of the graduation year. Gender was recorded in two categories only. Every filled-in age is our assumption, not the teenager's answer.

## Step 3: Choose and prepare the features

We cluster by interests only: the 36 word columns, from basketball to drugs. Then we scale them, as with the music: k-means compares teenagers by distance, and words that are used a lot would otherwise dominate. This time we use the *z-score*: how many standard deviations a value lies above or below the average of its column (https://www.statisticshowto.com/probability-and-statistics/z-score/). Type in:
```
from scipy.stats import zscore
interests = teens.loc[:, 'basketball':'drugs']
interests_z = interests.apply(zscore)
interests_z.head()
```

> **Background: why scale again?** (optional reading)
>
> k-means calculates distances between the observations and the centres of the clusters, so it is strongly influenced by the size of the values. Imagine one column with values from 1 to 10 and another from 1 to 1000: how could we compare distances between them? Our interests have very different ranges, as they count how many times a keyword appears on a profile.
>
> For the music, we normalised by hand to a range from 0 to 1. Here, we use the *z-score* from the package `scipy`: each value becomes its distance from the column's mean, measured in standard deviations. After scaling, every column has a mean of 0.

## Step 4: The model, k-means

k-means (MacQueen, 1967; https://en.wikipedia.org/wiki/K-means_clustering) is old but still widely used, because it is fast and simple. You stepped through it by hand in the code-along:
1. place k centres at random;
2. assign every observation to its nearest centre;
3. move each centre to the middle of its observations, and repeat until nothing changes.

We have to tell k-means how many clusters to find. Run the cell below to choose `k = 5`.

In [ ]:
#Keep cell

k = 5

Make a `KMeans` model called `teen_clusters` with `n_clusters = k` and `random_state=4711`, and fit it to `interests_z`. Then count the teenagers in each cluster with `np.bincount(teen_clusters.labels_)`. Type in:
```
from sklearn.cluster import KMeans
teen_clusters = KMeans(n_clusters = k, random_state=4711)
teen_clusters.fit(interests_z)
np.bincount(teen_clusters.labels_)
```

> **Background: k-means and choosing k** (optional reading)
>
> Despite being ancient in computer terms, k-means is still widely used, because it delivers good results with great *computational performance*: the effort it takes in computing resources, which you notice mainly as execution time. https://machinelearningmastery.com/clustering-algorithms-with-python/ gives an overview of other clustering algorithms in Python.
>
> Many decisions in analytics are more an art than a science. We often have to estimate settings from experience or background knowledge, and k-means is famous for depending heavily on `k`, the number of clusters.
>
> The results also depend on where k-means places its first centres, at random. With another `random_state`, you can get quite different clusters. Rerun k-means a few times and look at the cluster sizes: you especially want to avoid clusters with only a few members.

> **Who decided?** We chose k = 5, following Lantz. With another k, k-means finds other groups, and it always finds k groups, whether they exist or not. In the workshop, you will try to find a good k.

## Step 5: Interpret the clusters

The *centre* of each cluster is the average of its teenagers' (scaled) interests. Run the cell below: it shows the five interests that score highest in each cluster.

In [ ]:
#Keep cell
interests_centroids = pd.DataFrame(teen_clusters.cluster_centers_, columns=interests_z.columns)

for c in range(k):
    top = interests_centroids.loc[c].sort_values(ascending=False).head(5)
    print('Cluster', c, ':', list(top.index))

> **Background: cluster centres** (optional reading)
>
> The centres are stored in `teen_clusters.cluster_centers_`. As a dataframe, each row is a cluster and each column an interest. A value is the average (scaled) count of that word among the teenagers in the cluster: positive values mean more than the average teenager, negative values less.
>
> To read a cluster, we look for the interests with the highest values, as the cell above does. Pandas' `idxmax()` does the opposite: `interests_centroids.idxmax()` gives for each interest the cluster in which it scores highest.

Lantz (2013, p. 288) named his clusters like this. Do your clusters match? Do the names make sense to you? Do you remember all those teenage films?

![alt text](https://github.com/goto4711/cdai/blob/main/teen-clusters.png?raw=true "")

Now we can ask who is in each cluster. Add the cluster of each teenager to `teens` with `teens['cluster'] = teen_clusters.labels_`, and compute the average age, share of girls and number of friends per cluster with `teens.groupby('cluster')[['age', 'female', 'friends']].mean()`.

The clusters hardly differ in age. Most users are girls (81% of those whose gender is known), so they dominate every cluster, but more in some than in others. The number of friends differs most: interests seem to go together with how social a teenager is online. Note that `female` is empty where gender is missing, so its mean is the share of girls among those whose gender is known.

> **Who decided?** The machine only found groups of numbers. We named them, and the names, such as Lantz's 'princesses' or 'criminals', are ours. They also carry our ideas about teenagers.

---

## The steps, and who decided

| Step | Danceable songs (supervised) | Teenagers' interests (unsupervised) |
|---|---|---|
| 1. Collect | Spotify's API; features computed by Spotify | 30,000 profiles; 36 words chosen by researchers |
| 2. Label or clean | danceability ≥ 0.6, our threshold | ages 13–19 only; missing ages filled by graduation year |
| 3. Features | 13 features we chose, normalised to 0–1 | 36 interests, scaled with z-scores |
| 4. Model | a neural network: 2 layers, 20 and 10 units | k-means with k = 5 |
| 5. Evaluate or interpret | 79% accuracy against 55% for guessing | five groups, named by us |

The same steps come back in every machine-learning project, and in your group project. At each of them, people make choices that the machine then learns from.

### Check an agent

Ask Gemini to "cluster the teenagers in teens by their interests with k-means into five clusters and describe the clusters". Run its code in the cell below. Does it scale the interests? Which columns does it use? Does it do anything about the impossible ages? Do you get the same clusters? You can only check this because you did it yourself first.

In [ ]:
# Paste Gemini's code here, run it and compare